# finance / household / costs

**Intent**
- Outputs `finance_household_costs_normal_annual_dollars`, the normal annual cost.
- Outputs `finance_household_costs_essential_dollars`, the part that cannot be cut.
- Outputs `finance_household_costs_discretionary_pct`, the discretionary share.
- Outputs `finance_household_costs_irregular_{expected,p95}_dollars`, the expected and one-in-twenty irregular cost.
- Outputs `finance_household_costs_growth_{low,base,high}_pct`, cost growth to 2070 in percent per year.
- Gives constants from the same classified rows, so they reconcile.
- Lets spending be projected and a bad year provisioned for.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is seventy two months of synthetic spending by category and class, with one irregular expense a year.
- `training` also holds ten years of synthetic inflation and cost growth around 3.24 percent.
- Both are synthetic until the `bank` table and the real series are loaded.
- The `Bank` workbook on Drive holds transactions from 2021-05-01, each with a `Category` that the `Classifier` maps to a bucket.
- The `bank` plugin writes nothing to Postgres yet.

**Status**
- Growth is a placeholder, a lookup of three numbers.
- It carries no evaluation, so the golden replay is the gate.

**Future**
- Add a `bank` table to `wrangle`, or a loader for the year files, and `query_postgres` to `workbench.data`.
- Explore spend by category, and flag one-offs above a robust threshold of the category median.
- Classify categories as essential, committed, discretionary or irregular, and check them against past behaviour.
- Fit a compound Poisson per irregular category, with known schedules kept apart.
- Model growth as inflation plus a structural premium, both mean reverting, and simulate to 2070.
- Take compound rates, with low, base and high as percentiles, conditioned on ageing, energy and insurance drivers.
- Compare five year growth with the five year inflation mean, as the history is under five years.
- Fix the limits against real data, then register and review before any release.
- Add `household/income` for Age Pension, super and rent.
- Serve the numbers to the planning sheet through `wrangle` and an Apps Script function, which is not yet built.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/household/costs"
MODEL_NAME = "finance_household_costs"
TARGET = {"kind": "served"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "household_costs_training_2026-11-05_2026-11-05"
TRAINING_GROWTH = "household_costs_growth_training_2026-11-05_2026-11-05"
GOLDEN = "household_costs_golden"

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
CATEGORIES = {
    "groceries": ("essential", 24000.0),
    "utilities": ("essential", 9600.0),
    "insurance": ("committed", 7200.0),
    "school": ("committed", 7200.0),
    "hospitality": ("discretionary", 12000.0),
    "subscriptions": ("discretionary", 2400.0),
    "holidays": ("discretionary", 9600.0),
    "lifestyle": ("discretionary", 48000.0),
}
LIFESTYLE_DEVIATIONS = [-300.0, 200.0, -100.0, 300.0, -200.0, 0.0, 100.0, -300.0, 200.0, -100.0, 300.0, -100.0]
LUMPS = [12000.0, 30000.0, 8000.0, 15000.0, 52000.0, 9000.0]
INFLATION_DEVIATIONS = [-1.2, 0.8, -0.4, 1.0, -0.6, 0.2, 0.6, -1.0, 0.4, 0.2]
GROWTH_DEVIATIONS = [0.5, -0.7, 0.3, 0.9, -0.4, -0.8, 0.6, 0.2, -0.3, -0.3]


def spending():
    rows = {"month": [], "category": [], "class": [], "amount_dollars": []}
    for month in range(72):
        for category, (kind, annual) in CATEGORIES.items():
            rows["month"].append(month)
            rows["category"].append(category)
            rows["class"].append(kind)
            rows["amount_dollars"].append(annual / 12.0 + (LIFESTYLE_DEVIATIONS[month % 12] if category == "lifestyle" else 0.0))
        if month % 12 == 5:
            rows["month"].append(month)
            rows["category"].append("irregular")
            rows["class"].append("irregular")
            rows["amount_dollars"].append(LUMPS[month // 12])
    return rows


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, spending)
growth_pl, growth_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING_GROWTH,
    lambda: {
        "year": list(range(2016, 2026)),
        "inflation_pct": [round(3.24 + deviation, 4) for deviation in INFLATION_DEVIATIONS],
        "cost_growth_pct": [round(4.5 + deviation, 4) for deviation in GROWTH_DEVIATIONS],
    },
)
print("transactions", training_pl.height, "annual", growth_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
import numpy as np

month = training_pl["month"].to_numpy()
kind = training_pl["class"].to_numpy()
amount = training_pl["amount_dollars"].to_numpy()
latest = month >= month.max() - 11
regular = kind != "irregular"
NORMAL = round(float(amount[regular & latest].sum()), 0)
ESSENTIAL = round(float(amount[np.isin(kind, ["essential", "committed"]) & latest].sum()), 0)
DISCRETIONARY = round(float(amount[(kind == "discretionary") & latest].sum() / amount[regular & latest].sum() * 100.0), 2)
years = month // 12
lumps = np.array([amount[(kind == "irregular") & (years == year)].sum() for year in np.unique(years)])
INFLATION = round(float(growth_pl["inflation_pct"].mean()), 2)
GROWTH = round(float(growth_pl["cost_growth_pct"].mean()), 2)
CONSTANTS = dict(
    zip(
        OUTPUTS,
        [NORMAL, ESSENTIAL, DISCRETIONARY, round(float(lumps.mean()), 0), round(float(np.percentile(lumps, 95)), 0), INFLATION, GROWTH, round(2 * GROWTH - INFLATION, 2)],
        strict=True,
    )
)
totals = np.array([amount[month == m].sum() for m in np.unique(month)])
median = float(np.median(totals))
mad = float(np.median(np.abs(totals - median)))
flagged = np.flatnonzero(totals > median + 5.0 * mad).tolist()
print("flagged months", flagged, "constants", CONSTANTS)
assert flagged == sorted(set(month[kind == "irregular"].tolist())), "the robust rule and the irregular class disagree"
assert list(CONSTANTS.values()) == [120000.0, 48000.0, 60.0, 21000.0, 46500.0, 3.24, 4.5, 5.76], "the fit did not recover the placeholder"

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUTS[0]: [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic classified transactions and annual inflation and cost growth, training split",
        [training_path, growth_path, golden_path],
        "regular and irregular split of classified rows, and mean inflation and cost growth",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped loader")

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGET, metadata["version"])
print("registered version", version)